In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1OPjUy1LmbVFyct9QzTp0MdJv9qvA5MeK", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/05_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
import time
import math

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f"   Memory: {total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. This notebook will run on CPU (slower but functional).")
    print("   Go to Runtime → Change runtime type → GPU (recommended)")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Clean plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'figure.dpi': 100,
})

%matplotlib inline

# 🚀 Build & Run Mini-Qwen3.5 Locally

*Part 5 of the Vizuara series on Building Qwen3.5 from Scratch*
*Estimated time: ~120 minutes*

**This is the capstone notebook.** Over the past four notebooks, we built every piece of Qwen3.5's architecture from first principles:

1. **Notebook 1:** From Softmax to Linear Attention — the $O(N^2) \to O(N)$ trick
2. **Notebook 2:** DeltaNet — error-correcting memory via the delta rule
3. **Notebook 3:** Gated DeltaNet — exponential gating + delta rule for the full layer
4. **Notebook 4:** Hybrid Attention + MoE — the 3:1 pattern and fine-grained routing

Now we **put it all together**. In this notebook, we will:
1. **Build a complete Mini-Qwen3.5 model** — stacking all the components into a real language model
2. **Train it from scratch** on a text dataset and watch it learn to generate text
3. **Explore the full Qwen3.5 model family** — from 0.8B on smartphones to 397B on clusters
4. **Quantize the model to int8** — measuring size reduction and speed improvements
5. **Understand local deployment** — Ollama, llama.cpp, thinking modes, and KV cache savings

By the end, you will have trained your own tiny language model with Qwen3.5's architecture. Let us get started.

In [ ]:
#@title 🎧 Listen: Ai Assistant
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_02_ai_assistant.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/build-llm/practice/7/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

---


In [ ]:
#@title 🎧 Transition: Part A Recap Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_03_part_a_recap_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part A: The Complete Architecture — From Components to Model

### Recap: What We Built

Over four notebooks, we built these components:

| Component | What It Does | Notebook |
|-----------|-------------|----------|
| Linear Attention | $O(N)$ token mixing via the kernel trick | NB 1 |
| Delta Rule | Error-correcting state updates: $S_t = S_{t-1} + k(v - S^T k)^T$ | NB 2 |
| Gated DeltaNet | Exponential gating + delta rule for selective memory | NB 3 |
| Gated Attention | Softmax attention with a learned sigmoid gate | NB 4 |
| MoE FFN | Fine-grained routing: 10 of 512 experts + 1 shared | NB 4 |
| SwiGLU FFN | Dense feed-forward: $\text{SiLU}(xW_1) \odot xW_3) W_2$ | NB 4 |

Now we stack them into a complete language model. Here is the architecture:

```
Input Token IDs
       │
       ▼
┌─────────────┐
│  Embedding  │  (vocab_size → d_model)
└──────┬──────┘
       │
       ▼
┌─────────────────────────────────┐
│  Block 1: GatedDeltaNet + FFN  │ ◄─── Residual + RMSNorm
├─────────────────────────────────┤
│  Block 2: GatedDeltaNet + FFN  │ ◄─── Residual + RMSNorm
├─────────────────────────────────┤
│  Block 3: GatedDeltaNet + FFN  │ ◄─── Residual + RMSNorm
├─────────────────────────────────┤
│  Block 4: GatedAttention + FFN │ ◄─── Residual + RMSNorm (global refresh)
├─────────────────────────────────┤
│         ... repeat ...          │
└──────┬──────────────────────────┘
       │
       ▼
┌─────────────┐
│   RMSNorm   │  (final normalization)
└──────┬──────┘
       │
       ▼
┌─────────────┐
│   LM Head   │  (d_model → vocab_size)
└──────┬──────┘
       │
       ▼
  Logits (next-token prediction)
```


In [ ]:
#@title 🎧 Listen: Key Design Choices
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_04_key_design_choices.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Key Design Choices for Our Mini Model

We build a **simplified** version that captures the essential architecture:
- **Dense FFN** (SwiGLU) instead of MoE — keeps training simple
- **Simplified GatedDeltaNet** — the gated delta update without the full Conv1D pipeline
- **Character-level tokenizer** — no need to download a large tokenizer
- **Small dimensions** — fits in Colab's memory and trains in minutes

In [ ]:
#@title 🎧 Code Walkthrough: Building Blocks Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_05_building_blocks_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# BUILDING BLOCKS: RMSNorm, Rotary Position Embeddings, SwiGLU
# ============================================================

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization.

    Unlike LayerNorm, RMSNorm does not center the activations (no mean subtraction).
    This makes it slightly faster while being equally effective.

    Formula: output = x / RMS(x) * gamma
    where RMS(x) = sqrt(mean(x^2) + eps)
    """
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))  # Learnable scale (gamma)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight


class RotaryPositionEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE).

    Encodes position by rotating pairs of dimensions in the key/query vectors.
    This allows the model to learn relative positions naturally.
    """
    def __init__(self, d_head: int, max_seq_len: int = 2048, base: float = 10000.0):
        super().__init__()
        # Compute frequency bands: theta_i = 1 / base^(2i/d)
        inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
        self.register_buffer('inv_freq', inv_freq)

        # Precompute sin/cos for all positions
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)  # (max_seq_len, d_head/2)
        self.register_buffer('cos_cached', freqs.cos())  # (max_seq_len, d_head/2)
        self.register_buffer('sin_cached', freqs.sin())  # (max_seq_len, d_head/2)

    def forward(self, x: torch.Tensor, seq_len: int) -> torch.Tensor:
        """Apply rotary embeddings to x.

        Args:
            x: (batch, n_heads, seq_len, d_head)
            seq_len: actual sequence length
        Returns:
            Rotated x with position information encoded
        """
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)  # (1, 1, seq, d/2)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)

        # Split x into pairs and rotate
        x1, x2 = x[..., ::2], x[..., 1::2]
        rotated = torch.cat([
            x1 * cos - x2 * sin,
            x1 * sin + x2 * cos,
        ], dim=-1)
        return rotated


class SwiGLUFFN(nn.Module):
    """SwiGLU Feed-Forward Network.

    The dense FFN used in the Qwen3.5 smaller models:
    output = (SiLU(x @ W_gate) * (x @ W_up)) @ W_down

    SiLU(z) = z * sigmoid(z)
    """
    def __init__(self, d_model: int, d_ff: int = None):
        super().__init__()
        if d_ff is None:
            d_ff = int(d_model * 8 / 3)  # Standard ratio
            d_ff = ((d_ff + 63) // 64) * 64  # Round to multiple of 64
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# Quick test
print("=== Building Block Tests ===\n")

batch, seq, d = 2, 16, 256
x_test = torch.randn(batch, seq, d)

# RMSNorm
norm = RMSNorm(d)
normed = norm(x_test)
print(f"RMSNorm: input shape {x_test.shape} → output shape {normed.shape}")
rms_val = torch.sqrt(torch.mean(normed ** 2, dim=-1)).mean().item()
print(f"  Output RMS ≈ {rms_val:.3f} (should be close to 1.0)")

# RoPE
rope = RotaryPositionEmbedding(d_head=32, max_seq_len=512)
x_heads = torch.randn(batch, 8, seq, 32)
rotated = rope(x_heads, seq)
print(f"\nRoPE: input shape {x_heads.shape} → output shape {rotated.shape}")

# SwiGLU
ffn = SwiGLUFFN(d)
ffn_out = ffn(x_test)
print(f"\nSwiGLU FFN: input shape {x_test.shape} → output shape {ffn_out.shape}")
print(f"  FFN parameters: {sum(p.numel() for p in ffn.parameters()):,}")

print("\n✅ All building blocks working!")

In [ ]:
#@title 🎧 Code Walkthrough: Attention Layers Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_06_attention_layers_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# ATTENTION LAYERS: Simplified GatedDeltaNet & GatedAttention
# ============================================================

class SimplifiedGatedDeltaNet(nn.Module):
    """Simplified Gated DeltaNet for our mini model.

    Core update rule per head:
        S_t = alpha_t * S_{t-1} + beta_t * k_t (v_t - S_{t-1}^T k_t)^T

    This simplified version:
    - Uses learned linear projections for Q, K, V, alpha, beta
    - Applies L2 normalization to Q and K for stability
    - Processes tokens sequentially (recurrent mode)

    In the full Qwen3.5, there is also a Conv1D for local context,
    but we omit it here for clarity.
    """
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int = 512):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        # Projections for Q, K, V
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        # Gate projections (per-head scalar gates)
        self.alpha_proj = nn.Linear(d_model, n_heads, bias=True)  # Decay gate
        self.beta_proj = nn.Linear(d_model, n_heads, bias=True)   # Write gate

        # RoPE for positional encoding
        self.rope = RotaryPositionEmbedding(self.d_head, max_seq_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            output: (batch, seq_len, d_model)
        """
        B, T, D = x.shape
        H = self.n_heads
        d = self.d_head

        # Project to Q, K, V
        q = self.q_proj(x).view(B, T, H, d).transpose(1, 2)  # (B, H, T, d)
        k = self.k_proj(x).view(B, T, H, d).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, d).transpose(1, 2)

        # Apply RoPE
        q = self.rope(q, T)
        k = self.rope(k, T)

        # L2 normalize Q and K for stable training
        q = F.normalize(q, p=2, dim=-1)
        k = F.normalize(k, p=2, dim=-1)

        # Compute gates: alpha (decay), beta (write strength)
        alpha = torch.sigmoid(self.alpha_proj(x)).transpose(1, 2)  # (B, H, T)
        beta = torch.sigmoid(self.beta_proj(x)).transpose(1, 2)

        # Recurrent processing with gated delta rule
        S = torch.zeros(B, H, d, d, device=x.device, dtype=x.dtype)  # State matrix
        outputs = []

        for t in range(T):
            k_t = k[:, :, t, :]    # (B, H, d)
            v_t = v[:, :, t, :]
            q_t = q[:, :, t, :]
            a_t = alpha[:, :, t].unsqueeze(-1).unsqueeze(-1)  # (B, H, 1, 1)
            b_t = beta[:, :, t].unsqueeze(-1).unsqueeze(-1)

            # Delta rule: compute prediction error
            prediction = torch.einsum('bhij,bhj->bhi', S, k_t)  # S^T @ k
            error = v_t - prediction

            # Gated delta update: S = alpha * S + beta * outer(k, error)
            delta = torch.einsum('bhi,bhj->bhij', k_t, error)  # outer product
            S = a_t * S + b_t * delta

            # Query the state
            out_t = torch.einsum('bhij,bhj->bhi', S, q_t)  # S^T @ q
            outputs.append(out_t)

        # Stack outputs: (B, H, T, d) → (B, T, D)
        output = torch.stack(outputs, dim=2)  # (B, H, T, d)
        output = output.transpose(1, 2).contiguous().view(B, T, D)
        return self.o_proj(output)


class GatedAttention(nn.Module):
    """Gated Attention — standard softmax attention with a learned sigmoid gate.

    GatedAttn(Q, K, V) = sigmoid(g) * softmax(QK^T / sqrt(d)) V

    This is used for 25% of the layers (every 4th layer) to provide
    global, full-context retrieval as a "refresh" mechanism.
    """
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int = 512):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        # Sigmoid gate — the key difference from standard attention
        self.gate_proj = nn.Linear(d_model, d_model, bias=True)

        # RoPE
        self.rope = RotaryPositionEmbedding(self.d_head, max_seq_len)

        # Causal mask
        mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        self.register_buffer('causal_mask', mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        H = self.n_heads
        d = self.d_head

        q = self.q_proj(x).view(B, T, H, d).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, d).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, d).transpose(1, 2)

        # Apply RoPE
        q = self.rope(q, T)
        k = self.rope(k, T)

        # Scaled dot-product attention with causal mask
        scale = math.sqrt(d)
        attn = torch.matmul(q, k.transpose(-2, -1)) / scale  # (B, H, T, T)
        attn = attn.masked_fill(self.causal_mask[:T, :T].unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn_out = torch.matmul(attn, v)  # (B, H, T, d)

        # Apply sigmoid gate
        gate = torch.sigmoid(self.gate_proj(x))  # (B, T, D)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D)
        output = gate * attn_out

        return self.o_proj(output)


# Quick test
print("=== Attention Layer Tests ===\n")

B, T, D, H = 2, 32, 256, 8
x_test = torch.randn(B, T, D)

gdn = SimplifiedGatedDeltaNet(D, H, max_seq_len=512)
gdn_out = gdn(x_test)
print(f"GatedDeltaNet: {x_test.shape} → {gdn_out.shape}")
print(f"  Parameters: {sum(p.numel() for p in gdn.parameters()):,}")

ga = GatedAttention(D, H, max_seq_len=512)
ga_out = ga(x_test)
print(f"GatedAttention: {x_test.shape} → {ga_out.shape}")
print(f"  Parameters: {sum(p.numel() for p in ga.parameters()):,}")

print("\n✅ Both attention layers working!")

In [ ]:
#@title 🎧 Code Walkthrough: Qwen35 Block Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_07_qwen35_block_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# THE QWEN3.5 BLOCK: Attention → Add & Norm → FFN → Add & Norm
# ============================================================

class Qwen35Block(nn.Module):
    """A single Qwen3.5 Transformer block.

    Structure:
        h = RMSNorm(x + Attention(x))        # Pre-norm + residual
        output = RMSNorm(h + SwiGLU_FFN(h))   # Pre-norm + residual

    Args:
        d_model: Hidden dimension
        n_heads: Number of attention heads
        max_seq_len: Maximum sequence length
        use_gated_attention: If True, use GatedAttention; else GatedDeltaNet
    """
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int = 512,
                 use_gated_attention: bool = False):
        super().__init__()

        # Attention layer (3:1 hybrid pattern)
        if use_gated_attention:
            self.attn = GatedAttention(d_model, n_heads, max_seq_len)
        else:
            self.attn = SimplifiedGatedDeltaNet(d_model, n_heads, max_seq_len)

        # Feed-forward network (dense SwiGLU for our mini model)
        self.ffn = SwiGLUFFN(d_model)

        # Normalization layers (pre-norm architecture)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)

        self.use_gated_attention = use_gated_attention

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm attention + residual
        h = x + self.attn(self.norm1(x))
        # Pre-norm FFN + residual
        output = h + self.ffn(self.norm2(h))
        return output


# Test the block
block_dn = Qwen35Block(256, 8, use_gated_attention=False)
block_ga = Qwen35Block(256, 8, use_gated_attention=True)

x_test = torch.randn(2, 32, 256)
print(f"GatedDeltaNet block: {x_test.shape} → {block_dn(x_test).shape}")
print(f"GatedAttention block: {x_test.shape} → {block_ga(x_test).shape}")
print(f"\n✅ Qwen3.5 block working!")

---


In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_08_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 🔧 TODO #1: Build the Complete Mini-Qwen3.5 Model

Now it is your turn to assemble the full model. You need to build `MiniQwen35` with:

1. **Token embedding** — maps vocabulary indices to d_model-dimensional vectors
2. **Stack of Qwen3.5 blocks** — following the 3:1 hybrid pattern (layers 0,1,2 = GatedDeltaNet, layer 3 = GatedAttention, then repeat)
3. **Final RMSNorm** — normalize before the output projection
4. **LM head** — project from d_model back to vocab_size (logits for next-token prediction)

The forward pass takes token IDs and returns logits of shape `(batch, seq_len, vocab_size)`.

**Mini-Qwen3.5 Configuration:**
- `d_model = 256` (hidden dimension)
- `n_heads = 8` (attention heads, so d_head = 32)
- `n_layers = 8` (2 repeating units of the 3:1 pattern)
- `vocab_size = 256` (character-level: one entry per byte)
- `max_seq_len = 512`

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_09_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Build the complete Mini-Qwen3.5 language model.
#
# Requirements:
#   1. self.embedding = nn.Embedding(vocab_size, d_model)
#   2. self.blocks = nn.ModuleList of n_layers Qwen35Block instances
#      - Use the 3:1 pattern: layers 0,1,2 → GatedDeltaNet; layer 3 → GatedAttention; repeat
#      - Hint: use_gated_attention = ((layer_idx + 1) % 4 == 0)
#   3. self.final_norm = RMSNorm(d_model)
#   4. self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
#   5. Weight tying: self.lm_head.weight = self.embedding.weight
#
#   Forward pass: embedding → blocks → final_norm → lm_head → logits
# ============ TODO ============

class MiniQwen35(nn.Module):
    """A mini Qwen3.5 language model for learning and experimentation.

    Architecture:
    - Character-level tokenization (vocab_size = 256)
    - 3:1 hybrid attention pattern (GatedDeltaNet : GatedAttention)
    - SwiGLU feed-forward networks
    - RMSNorm pre-normalization
    - Weight-tied embedding and LM head
    """

    def __init__(self, vocab_size: int = 256, d_model: int = 256,
                 n_heads: int = 8, n_layers: int = 8, max_seq_len: int = 512):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size

        # YOUR CODE: Create the 4 components listed above
        # 1. Token embedding
        self.embedding = ???

        # 2. Stack of Qwen3.5 blocks (3:1 hybrid pattern)
        self.blocks = nn.ModuleList([
            ???  # Hint: Qwen35Block(d_model, n_heads, max_seq_len, use_gated_attention=???)
            for i in range(n_layers)
        ])

        # 3. Final normalization
        self.final_norm = ???

        # 4. LM head (with weight tying)
        self.lm_head = ???
        self.lm_head.weight = self.embedding.weight  # Weight tying

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: (batch, seq_len) — integer token IDs
        Returns:
            logits: (batch, seq_len, vocab_size) — next-token prediction scores
        """
        # YOUR CODE: Forward pass through embedding → blocks → norm → lm_head
        x = ???  # Embed tokens
        for block in self.blocks:
            x = ???  # Pass through each block
        x = ???  # Final normalization
        logits = ???  # Project to vocabulary
        return logits

    def count_parameters(self) -> int:
        """Return total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
#@title 🎧 Code Walkthrough: Todo1 Verification Pre
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_10_todo1_verification_pre.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============

try:
    # Build the model
    model = MiniQwen35(
        vocab_size=256,
        d_model=256,
        n_heads=8,
        n_layers=8,
        max_seq_len=512,
    )

    # Test forward pass
    batch_size, seq_len = 2, 64
    test_ids = torch.randint(0, 256, (batch_size, seq_len))
    logits = model(test_ids)

    print("=== Mini-Qwen3.5 Verification ===\n")
    print(f"Input shape:  {test_ids.shape}  (batch={batch_size}, seq_len={seq_len})")
    print(f"Output shape: {logits.shape}  (should be [{batch_size}, {seq_len}, 256])")
    assert logits.shape == (batch_size, seq_len, 256), \
        f"Expected shape ({batch_size}, {seq_len}, 256), got {logits.shape}"


In [ ]:
#@title 🎧 Narration: Todo1 Verification Post
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_11_todo1_verification_post.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
    print(f"✅ Output shape correct!")

    # Check parameter count
    n_params = model.count_parameters()
    print(f"\nTotal parameters: {n_params:,}")
    print(f"  That's {n_params / 1e6:.1f}M parameters")

    # Check 3:1 pattern
    print(f"\nLayer pattern:")
    for i, block in enumerate(model.blocks):
        layer_type = "GatedAttention" if block.use_gated_attention else "GatedDeltaNet"
        print(f"  Layer {i}: {layer_type}")

    n_deltanet = sum(1 for b in model.blocks if not b.use_gated_attention)
    n_gattn = sum(1 for b in model.blocks if b.use_gated_attention)
    print(f"\n  GatedDeltaNet layers: {n_deltanet}")
    print(f"  GatedAttention layers: {n_gattn}")
    assert n_deltanet == 6 and n_gattn == 2, \
        f"Expected 6 DeltaNet + 2 GatedAttention for 8 layers in 3:1 pattern"
    print(f"  ✅ 3:1 hybrid pattern confirmed!")

    # Check weight tying
    assert model.lm_head.weight is model.embedding.weight, \
        "LM head and embedding weights should be tied!"
    print(f"\n✅ Weight tying confirmed!")
    print(f"\n✅ All checks passed! Your Mini-Qwen3.5 is ready to train.")

except NameError:
    print("❌ Replace the ??? placeholders with your code.")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback; traceback.print_exc()

---


In [ ]:
#@title 🎧 Narration: Stop And Think1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_12_stop_and_think1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### ✋ Stop and Think
Before looking at the solution:
1. Why do we use weight tying between the embedding and the LM head? What does this assume about the relationship between "understanding a token" and "predicting a token"?
2. With 8 layers in a 3:1 pattern, which layers are GatedAttention? (Hint: layers where `(index + 1) % 4 == 0`)
3. What is the purpose of the final RMSNorm before the LM head?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Solution1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_13_solution1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
class MiniQwen35(nn.Module):
    """A mini Qwen3.5 language model for learning and experimentation."""

    def __init__(self, vocab_size: int = 256, d_model: int = 256,
                 n_heads: int = 8, n_layers: int = 8, max_seq_len: int = 512):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size

        # 1. Token embedding
        self.embedding = nn.Embedding(vocab_size, d_model)

        # 2. Stack of Qwen3.5 blocks (3:1 hybrid pattern)
        self.blocks = nn.ModuleList([
            Qwen35Block(
                d_model, n_heads, max_seq_len,
                use_gated_attention=((i + 1) % 4 == 0),  # Every 4th layer
            )
            for i in range(n_layers)
        ])

        # 3. Final normalization
        self.final_norm = RMSNorm(d_model)

        # 4. LM head (with weight tying)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(input_ids)                    # (B, T) → (B, T, D)
        for block in self.blocks:
            x = block(x)                                  # (B, T, D)
        x = self.final_norm(x)                            # (B, T, D)
        logits = self.lm_head(x)                          # (B, T, vocab_size)
        return logits

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int = 100,
                 temperature: float = 0.8, top_k: int = 50) -> torch.Tensor:
        """Autoregressive text generation."""
        for _ in range(max_new_tokens):
            # Crop to max_seq_len if needed
            idx_cond = input_ids[:, -512:]
            logits = self.forward(idx_cond)
            logits = logits[:, -1, :] / temperature  # Last token logits

            # Top-k sampling
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)

        return input_ids


# Build and verify
model = MiniQwen35(vocab_size=256, d_model=256, n_heads=8, n_layers=8, max_seq_len=512)
n_params = model.count_parameters()

print(f"Mini-Qwen3.5 built successfully!")
print(f"  Parameters: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Layers: {len(model.blocks)}")
print(f"  Pattern: ", end="")
for i, b in enumerate(model.blocks):
    tag = "GA" if b.use_gated_attention else "DN"
    print(tag, end=" ")
print(f"\n  (DN = GatedDeltaNet, GA = GatedAttention)")

# Test generation (random weights — output will be gibberish)
test_input = torch.tensor([[72, 101, 108, 108, 111]])  # "Hello" in ASCII
generated = model.generate(test_input, max_new_tokens=20, temperature=1.0)
decoded = ''.join(chr(min(c, 127)) for c in generated[0].tolist())
print(f"\nRandom generation test (untrained): '{decoded}'")
print(f"  (Gibberish expected — we have not trained yet!)")

print(f"\n✅ Model is ready for training!")

---


In [ ]:
#@title 🎧 Transition: Part B Config Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_14_part_b_config_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part B: Mini-Qwen3.5 Configuration

Here is our mini model compared to the real Qwen3.5 family:

| Parameter | Mini (ours) | Qwen3.5-0.8B | Qwen3.5-9B | Qwen3.5-397B |
|-----------|-------------|---------------|-------------|---------------|
| d_model | 256 | 1,024 | 3,584 | 4,096 |
| n_heads | 8 | 16 | 28 | 64 |
| n_layers | 8 | 24 | 36 | 60 |
| Vocab size | 256 (char) | 248,320 | 248,320 | 248,320 |
| Context length | 512 | 32K | 262K | 262K |
| FFN type | Dense (SwiGLU) | Dense | Dense | MoE (512 experts) |
| Total params | ~5M | 0.8B | 9B | 397B |
| Active params | ~5M | 0.8B | 9B | 17B |

Our mini model captures the **architecture** faithfully — the 3:1 hybrid pattern, gated updates, RMSNorm, SwiGLU, RoPE, weight tying — just at a much smaller scale.

In [ ]:
#@title 🎧 What to Look For: Architecture Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_15_architecture_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the architecture — layer-by-layer parameter breakdown
print("=" * 65)
print("  MINI-QWEN3.5 ARCHITECTURE SUMMARY")
print("=" * 65)

print("""
┌─────────────────────────────────────────────────────────────┐
│                    Mini-Qwen3.5                             │
│                                                             │
│  Input: Token IDs (batch, seq_len)                          │
│         ↓                                                   │
│  ┌──────────────────────────────────────────────────────┐   │
│  │  Embedding: 256 tokens → 256 dims                    │   │
│  └──────────────────────┬───────────────────────────────┘   │
│                         ↓                                   │
│  ┌──────────────────────────────────────────────────────┐   │
│  │  Layer 0: GatedDeltaNet → Add&Norm → SwiGLU → Add&Norm│  │
│  │  Layer 1: GatedDeltaNet → Add&Norm → SwiGLU → Add&Norm│  │
│  │  Layer 2: GatedDeltaNet → Add&Norm → SwiGLU → Add&Norm│  │
│  │  Layer 3: GatedAttention→ Add&Norm → SwiGLU → Add&Norm│  │
│  │  ─── repeating unit ──────────────────────────────── │   │
│  │  Layer 4: GatedDeltaNet → Add&Norm → SwiGLU → Add&Norm│  │
│  │  Layer 5: GatedDeltaNet → Add&Norm → SwiGLU → Add&Norm│  │
│  │  Layer 6: GatedDeltaNet → Add&Norm → SwiGLU → Add&Norm│  │
│  │  Layer 7: GatedAttention→ Add&Norm → SwiGLU → Add&Norm│  │
│  └──────────────────────┬───────────────────────────────┘   │
│                         ↓                                   │
│  ┌──────────────────────────────────────────────────────┐   │
│  │  Final RMSNorm → LM Head (256 dims → 256 vocab)     │   │
│  └──────────────────────┬───────────────────────────────┘   │
│                         ↓                                   │
│  Output: Logits (batch, seq_len, vocab_size)                │
└─────────────────────────────────────────────────────────────┘
""")

# Parameter breakdown
print("\n📊 Parameter Breakdown:")
param_groups = {
    'Embedding (+ LM head, tied)': sum(p.numel() for p in model.embedding.parameters()),
    'Final RMSNorm': sum(p.numel() for p in model.final_norm.parameters()),
}

dn_params = 0
ga_params = 0
ffn_params = 0
norm_params = 0

for block in model.blocks:
    attn_p = sum(p.numel() for p in block.attn.parameters())
    ffn_p = sum(p.numel() for p in block.ffn.parameters())
    n_p = sum(p.numel() for p in block.norm1.parameters()) + \
          sum(p.numel() for p in block.norm2.parameters())

    if block.use_gated_attention:
        ga_params += attn_p
    else:
        dn_params += attn_p
    ffn_params += ffn_p
    norm_params += n_p

param_groups['GatedDeltaNet layers (×6)'] = dn_params
param_groups['GatedAttention layers (×2)'] = ga_params
param_groups['SwiGLU FFN layers (×8)'] = ffn_params
param_groups['RMSNorm layers (×16 + final)'] = norm_params + param_groups['Final RMSNorm']

total = sum(param_groups.values())
for name, count in param_groups.items():
    pct = 100 * count / total
    bar = '█' * int(pct / 2) + '░' * (50 - int(pct / 2))
    print(f"  {name:<35} {count:>10,} ({pct:>5.1f}%) {bar}")

print(f"  {'─' * 35} {'─' * 10}")
print(f"  {'TOTAL':<35} {total:>10,}")

---


In [ ]:
#@title 🎧 Transition: Part C Training Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_16_part_c_training_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part C: Training on Real Text

Now let us train our Mini-Qwen3.5! We will use a text dataset and watch the model learn to predict the next character.

### Training Setup
- **Data**: Shakespeare's works (a classic for character-level language models)
- **Tokenization**: Character-level (each byte = one token, vocab_size = 256)
- **Optimizer**: AdamW with learning rate warmup
- **Steps**: 200 training steps (enough to see learning with clear loss decrease)

In [ ]:
#@title 🎧 Code Walkthrough: Data Prep Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_17_data_prep_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# DATA PREPARATION: Download Shakespeare text
# ============================================================
import urllib.request

# Try to download Shakespeare. Fall back to synthetic text.
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

try:
    print("📥 Downloading Shakespeare text...")
    response = urllib.request.urlopen(DATA_URL, timeout=10)
    text = response.read().decode('utf-8')
    print(f"✅ Downloaded {len(text):,} characters of Shakespeare")
except Exception as e:
    print(f"⚠️ Download failed ({e}). Using synthetic text.")
    # Generate synthetic training data
    patterns = [
        "To be or not to be, that is the question. ",
        "All that glitters is not gold. ",
        "The quick brown fox jumps over the lazy dog. ",
        "Now is the winter of our discontent. ",
        "A rose by any other name would smell as sweet. ",
        "What light through yonder window breaks? ",
        "Friends, Romans, countrymen, lend me your ears. ",
        "The course of true love never did run smooth. ",
        "We are such stuff as dreams are made on. ",
        "Brevity is the soul of wit. ",
    ]
    text = ""
    while len(text) < 500_000:
        text += random.choice(patterns)
    print(f"✅ Generated {len(text):,} characters of synthetic text")

# Character-level tokenization: each character → its byte value
def encode(s: str) -> list:
    return [ord(c) for c in s if ord(c) < 256]

def decode(tokens: list) -> str:
    return ''.join(chr(t) for t in tokens if 0 <= t < 256)

# Encode the entire text
data = torch.tensor(encode(text), dtype=torch.long)
print(f"\nEncoded to {len(data):,} tokens")
print(f"Vocabulary: {len(set(data.tolist()))} unique characters used")

# Train/val split
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
val_data = data[split_idx:]
print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

# Sample
sample = decode(data[:200].tolist())
print(f"\nFirst 200 characters:\n{'─' * 40}\n{sample}\n{'─' * 40}")

In [ ]:
#@title 🎧 Code Walkthrough: Data Loader Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_18_data_loader_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# DATA LOADER: Create batches of (input, target) pairs
# ============================================================

def get_batch(split: str, batch_size: int = 16, seq_len: int = 128):
    """Get a random batch of input-target pairs for training.

    For language modeling, the target is the input shifted by one position:
        input:  [t0, t1, t2, ..., t_{n-1}]
        target: [t1, t2, t3, ..., t_n]
    """
    data_source = train_data if split == 'train' else val_data

    # Random starting positions
    max_start = len(data_source) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))

    # Build batches
    x = torch.stack([data_source[s : s + seq_len] for s in starts])
    y = torch.stack([data_source[s + 1 : s + seq_len + 1] for s in starts])

    return x.to(device), y.to(device)

# Test the data loader
x_batch, y_batch = get_batch('train', batch_size=4, seq_len=32)
print(f"Input batch shape:  {x_batch.shape}")
print(f"Target batch shape: {y_batch.shape}")
print(f"\nExample (first sequence):")
print(f"  Input:  '{decode(x_batch[0].tolist())}'")
print(f"  Target: '{decode(y_batch[0].tolist())}'")
print(f"\nNotice the target is shifted by 1 character — the model learns to predict the next char.")

---


In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_19_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 🔧 TODO #2: Training Loop

Write a training loop that:
1. Moves the model to the GPU (or CPU)
2. Uses AdamW optimizer with learning rate `3e-4` and weight decay `0.01`
3. Implements a simple linear warmup for the first 20 steps
4. Trains for 200 steps, logging loss every 10 steps
5. Evaluates on the validation set every 50 steps
6. After training, generates a sample of text to verify the model learned something
7. Plots the training loss curve

The cross-entropy loss should decrease from ~5.5 (random, since ln(256) ≈ 5.5) down to around 1.5-2.5 within 200 steps.

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_20_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Write the training loop for Mini-Qwen3.5.
#
# Steps:
#   1. model.to(device)
#   2. optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
#   3. For each step in range(200):
#      a. Get a batch: x, y = get_batch('train', batch_size=16, seq_len=128)
#      b. Forward pass: logits = model(x)
#      c. Compute loss: F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
#      d. Backward pass: loss.backward()
#      e. Optional: gradient clipping: torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#      f. optimizer.step(), optimizer.zero_grad()
#      g. Learning rate warmup: for first 20 steps, scale lr linearly from 0 to 3e-4
#      h. Log loss every 10 steps
#   4. After training, generate text with model.generate()
#   5. Plot the loss curve
# ============ TODO ============

# Move model to device
model = MiniQwen35(vocab_size=256, d_model=256, n_heads=8, n_layers=8, max_seq_len=512)
model = model.to(device)
print(f"Model on {device}")
print(f"Parameters: {model.count_parameters():,}")

# Training hyperparameters
N_STEPS = 200
BATCH_SIZE = 16
SEQ_LEN = 128
LR = 3e-4
WARMUP_STEPS = 20

# YOUR CODE: Create optimizer
optimizer = ???

# Training loop
train_losses = []
val_losses = []

print(f"\n{'Step':>6} {'Train Loss':>12} {'Val Loss':>12} {'LR':>10} {'Time':>8}")
print("─" * 54)

start_time = time.time()

for step in range(N_STEPS):
    model.train()

    # YOUR CODE: Learning rate warmup
    # For the first WARMUP_STEPS steps, scale lr from 0 to LR linearly
    # After warmup, keep lr at LR
    if step < WARMUP_STEPS:
        lr = ???  # Linear warmup
    else:
        lr = LR
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # YOUR CODE: Forward pass
    x, y = get_batch('train', batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
    logits = ???  # model forward pass
    loss = ???    # cross-entropy loss

    # YOUR CODE: Backward pass
    ???  # zero gradients
    ???  # backward
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    ???  # optimizer step

    train_losses.append(loss.item())

    # Logging
    if (step + 1) % 10 == 0 or step == 0:
        # Evaluate on validation set
        model.eval()
        with torch.no_grad():
            vx, vy = get_batch('val', batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
            vlogits = model(vx)
            vloss = F.cross_entropy(vlogits.view(-1, 256), vy.view(-1))
            val_losses.append((step, vloss.item()))

        elapsed = time.time() - start_time
        print(f"{step+1:>6} {loss.item():>12.4f} {vloss.item():>12.4f} {lr:>10.6f} {elapsed:>7.1f}s")


In [ ]:
#@title 🎧 Narration: Todo2 Post Training
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_21_todo2_post_training.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"\n{'─' * 54}")
print(f"Training complete in {time.time() - start_time:.1f}s")

# YOUR CODE: Generate sample text after training
model.eval()
prompt = "The "
prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
generated_ids = ???  # model.generate(prompt_ids, max_new_tokens=200, temperature=0.8)
generated_text = decode(generated_ids[0].tolist())

print(f"\n📝 Generated text (prompt: '{prompt}'):")
print(f"{'─' * 50}")
print(generated_text)
print(f"{'─' * 50}")

# YOUR CODE: Plot training loss
fig, ax = plt.subplots(figsize=(10, 5))
???  # Plot train_losses and val_losses
plt.show()

---


In [ ]:
#@title 🎧 Narration: Stop And Think2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_22_stop_and_think2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### ✋ Stop and Think
Before looking at the solution:
1. Why do we use learning rate warmup? What can go wrong if we start with a high learning rate?
2. The random loss for a 256-token vocabulary is $\ln(256) \approx 5.5$. If our model gets to loss 2.0, what does that mean in terms of bits per character?
3. Why do we clip gradients to a maximum norm of 1.0?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Solution2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_23_solution2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
# Full training loop with all components

model = MiniQwen35(vocab_size=256, d_model=256, n_heads=8, n_layers=8, max_seq_len=512)
model = model.to(device)
print(f"Model on {device}, Parameters: {model.count_parameters():,}")

# Hyperparameters
N_STEPS = 200
BATCH_SIZE = 16
SEQ_LEN = 128
LR = 3e-4
WARMUP_STEPS = 20

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

train_losses = []
val_losses = []

print(f"\n{'Step':>6} {'Train Loss':>12} {'Val Loss':>12} {'LR':>10} {'Time':>8}")
print("─" * 54)

start_time = time.time()

for step in range(N_STEPS):
    model.train()

    # Learning rate warmup
    if step < WARMUP_STEPS:
        lr = LR * (step + 1) / WARMUP_STEPS
    else:
        lr = LR
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Forward pass
    x, y = get_batch('train', batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, 256), y.view(-1))

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    train_losses.append(loss.item())

    # Logging
    if (step + 1) % 10 == 0 or step == 0:
        model.eval()
        with torch.no_grad():
            vx, vy = get_batch('val', batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
            vlogits = model(vx)
            vloss = F.cross_entropy(vlogits.view(-1, 256), vy.view(-1))
            val_losses.append((step, vloss.item()))

        elapsed = time.time() - start_time
        print(f"{step+1:>6} {loss.item():>12.4f} {vloss.item():>12.4f} {lr:>10.6f} {elapsed:>7.1f}s")

print(f"\n{'─' * 54}")
print(f"Training complete in {time.time() - start_time:.1f}s")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Started from: {train_losses[0]:.4f} (random: ln(256) ≈ {math.log(256):.2f})")

# Generate text
model.eval()
prompt = "The "
prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
generated_ids = model.generate(prompt_ids, max_new_tokens=200, temperature=0.8)
generated_text = decode(generated_ids[0].tolist())

print(f"\n📝 Generated text (prompt: '{prompt}'):")
print(f"{'─' * 50}")
print(generated_text[:300])
print(f"{'─' * 50}")
print("(The text will not be perfect Shakespeare, but it should show")
print(" recognizable English words and patterns — proof the model is learning!)")

# Plot training loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Training loss
axes[0].plot(range(1, len(train_losses) + 1), train_losses,
             color='#1565c0', linewidth=1.5, alpha=0.7, label='Train loss')
axes[0].axhline(y=math.log(256), color='gray', linestyle='--', alpha=0.5,
                label=f'Random baseline (ln(256) = {math.log(256):.2f})')
if val_losses:
    val_steps, val_vals = zip(*val_losses)
    axes[0].plot([s+1 for s in val_steps], val_vals, 'o-',
                 color='#e53935', linewidth=2, markersize=5, label='Val loss')
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('Mini-Qwen3.5 Training Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# Right: Loss in bits per character
bpc = [l / math.log(2) for l in train_losses]
axes[1].plot(range(1, len(bpc) + 1), bpc, color='#1565c0', linewidth=1.5, alpha=0.7)
axes[1].axhline(y=math.log(256) / math.log(2), color='gray', linestyle='--', alpha=0.5,
                label=f'Random ({math.log(256)/math.log(2):.1f} bpc)')
axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('Bits per Character', fontsize=12)
axes[1].set_title('Training Progress (bits per character)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n✅ Training complete! Loss decreased from {train_losses[0]:.2f} to {train_losses[-1]:.2f}")

---


In [ ]:
#@title 🎧 Transition: Part D Family Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_24_part_d_family_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part D: The Qwen3.5 Model Family

Our mini model has ~5M parameters. The real Qwen3.5 family spans from 0.8 billion to 397 billion parameters. Let us visualize the complete family and understand the scale of what these models achieve.

In [ ]:
#@title 🎧 What to Look For: Model Family Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_25_model_family_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# THE QWEN3.5 MODEL FAMILY
# ============================================================
import pandas as pd

# Model family data
family_data = {
    'Model': [
        'Mini (ours)', 'Qwen3.5-0.8B', 'Qwen3.5-2B', 'Qwen3.5-4B',
        'Qwen3.5-9B', 'Qwen3.5-35B-A3B', 'Qwen3.5-122B-A10B', 'Qwen3.5-397B-A17B'
    ],
    'Total Params': ['5M', '0.8B', '2B', '4B', '9B', '35B', '122B', '397B'],
    'Active Params': ['5M', '0.8B', '2B', '4B', '9B', '3B (MoE)', '10B (MoE)', '17B (MoE)'],
    'Memory (Q4)': ['~20 MB', '~1.6 GB', '~2.5 GB', '~3.5 GB', '~5 GB', '~20 GB', '~70 GB', '~230 GB'],
    'Runs On': [
        'This notebook!',
        'Smartphones, Raspberry Pi',
        'Mobile devices',
        'Laptops',
        'Standard laptops',
        'Single consumer GPU',
        'Multi-GPU server',
        'GPU cluster'
    ],
    'FFN Type': ['Dense', 'Dense', 'Dense', 'Dense', 'Dense', 'MoE', 'MoE', 'MoE'],
}

df = pd.DataFrame(family_data)

# Style the dataframe
print("=" * 80)
print("  THE QWEN3.5 MODEL FAMILY")
print("=" * 80)
print()

# Display with nice formatting
for _, row in df.iterrows():
    marker = "⭐" if row['Model'] == 'Mini (ours)' else "  "
    moe_tag = " [MoE]" if "MoE" in str(row['Active Params']) else ""
    print(f"{marker} {row['Model']:<22} | {row['Total Params']:>6} params | "
          f"{row['Memory (Q4)']:>8} | {row['Runs On']}{moe_tag}")

print()
print("Key insight: the MoE models have MANY parameters (knowledge capacity)")
print("but activate only a FRACTION per token (inference cost).")
print("Qwen3.5-397B has 397B params but runs like a 17B model.")

# Visualization: parameter count comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Total vs Active parameters
models = ['0.8B', '2B', '4B', '9B', '35B', '122B', '397B']
total_params = [0.8, 2, 4, 9, 35, 122, 397]
active_params = [0.8, 2, 4, 9, 3, 10, 17]

x = np.arange(len(models))
width = 0.35

bars1 = axes[0].bar(x - width/2, total_params, width, color='#1565c0',
                     alpha=0.8, label='Total Parameters', edgecolor='white')
bars2 = axes[0].bar(x + width/2, active_params, width, color='#e53935',
                     alpha=0.8, label='Active Parameters', edgecolor='white')

axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('Parameters (Billions)', fontsize=12)
axes[0].set_title('Qwen3.5: Total vs Active Parameters', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, fontsize=10)
axes[0].legend(fontsize=10)
axes[0].set_yscale('log')
axes[0].set_ylim(0.5, 500)

# Annotate MoE models
for i in range(4, 7):
    ratio = active_params[i] / total_params[i] * 100
    axes[0].annotate(f'{ratio:.1f}%\nactive',
                     xy=(x[i] + width/2, active_params[i]),
                     xytext=(0, 10), textcoords='offset points',
                     ha='center', fontsize=8, color='#e53935', fontweight='bold')

# Right: Memory footprint at Q4 quantization
memory_gb = [1.6, 2.5, 3.5, 5, 20, 70, 230]
colors = ['#4caf50', '#4caf50', '#4caf50', '#4caf50', '#ff9800', '#e53935', '#b71c1c']

bars = axes[1].barh(models, memory_gb, color=colors, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Memory at Q4 Quantization (GB)', fontsize=12)
axes[1].set_title('Model Size After 4-bit Quantization', fontsize=14, fontweight='bold')

# Add hardware annotations
hw_labels = ['Phone', 'Mobile', 'Laptop', 'Laptop', '1 GPU', 'Multi-GPU', 'Cluster']
for i, (bar, label) in enumerate(zip(bars, hw_labels)):
    axes[1].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                 label, va='center', fontsize=9, color='gray')

# Reference lines
axes[1].axvline(x=8, color='gray', linestyle='--', alpha=0.5)
axes[1].text(8.5, 6.5, '8 GB\n(laptop RAM)', fontsize=8, color='gray')
axes[1].axvline(x=24, color='gray', linestyle='--', alpha=0.5)
axes[1].text(24.5, 6.5, '24 GB\n(RTX 4090)', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

---


In [ ]:
#@title 🎧 Transition: Part E Quantization Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_26_part_e_quantization_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part E: Quantization — Making Models Small

### What Is Quantization?

Neural network weights are typically stored as 32-bit (FP32) or 16-bit (FP16/BF16) floating-point numbers. **Quantization** reduces the precision — for example, to 8-bit integers (INT8) or even 4-bit integers (INT4).

The intuition: most weight values cluster in a narrow range. We do not need 32 bits of precision to represent them. By mapping the range to fewer bits, we dramatically reduce model size with minimal quality loss.

| Precision | Bits per Weight | 9B Model Size | Quality |
|-----------|----------------|---------------|---------|
| FP32 | 32 | ~36 GB | Baseline |
| FP16/BF16 | 16 | ~18 GB | ≈ Identical |
| INT8 | 8 | ~9 GB | Very close |
| Q4_K_M (GGUF) | 4-ish | ~5 GB | Slight loss |

The GGUF Q4_K_M format used by llama.cpp is particularly clever — it uses a mix of 4-bit and 6-bit quantization for different weight matrices, preserving quality in the most sensitive layers.

Let us try dynamic INT8 quantization on our mini model and measure the impact.

---


In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_27_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 🔧 TODO #3: Quantize and Measure

Implement quantization analysis:
1. **Measure the original model size** (sum of all parameter memory in bytes)
2. **Apply dynamic INT8 quantization** using `torch.quantization.quantize_dynamic`
3. **Measure the quantized model size**
4. **Compare inference speed** (forward pass time for 100 sequences)
5. **Compare generation quality** (generate text from both models)
6. **Plot the comparison** (bar chart of size and speed)

In [ ]:
#@title 🎧 Before You Start: Todo3 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_28_todo3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Quantize the trained model and measure the impact.
#
# Steps:
#   1. Compute original model size in MB:
#      size_mb = sum(p.nelement() * p.element_size() for p in model.parameters()) / 1e6
#
#   2. Apply dynamic quantization (on CPU — quantization doesn't work on GPU):
#      model_cpu = model.cpu()
#      model_q = torch.quantization.quantize_dynamic(
#          model_cpu, {nn.Linear}, dtype=torch.qint8
#      )
#
#   3. Compute quantized model size (iterate over state_dict):
#      sum(v.nelement() * v.element_size() for v in model_q.state_dict().values()) / 1e6
#
#   4. Measure inference speed: time 100 forward passes for each model
#
#   5. Generate text from both models and compare
#
#   6. Create a bar chart comparing size and speed
# ============ TODO ============

# Move model to CPU for quantization
model_cpu = model.cpu()
model_cpu.eval()

# YOUR CODE: Step 1 — Original model size
original_size_mb = ???
print(f"Original model size: {original_size_mb:.2f} MB")

# YOUR CODE: Step 2 — Apply INT8 quantization
model_q = ???
print(f"✅ Quantization applied!")

# YOUR CODE: Step 3 — Quantized model size
quantized_size_mb = ???
print(f"Quantized model size: {quantized_size_mb:.2f} MB")
print(f"Compression ratio: {original_size_mb / quantized_size_mb:.2f}x")

# YOUR CODE: Step 4 — Speed comparison
# Run 100 forward passes with each model and measure total time
n_runs = 100
test_input = torch.randint(0, 256, (1, 64))

# Time original model
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        _ = model_cpu(test_input)
original_time = ???  # time.time() - start

# Time quantized model
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        _ = model_q(test_input)
quantized_time = ???  # time.time() - start

print(f"\nInference speed ({n_runs} runs, seq_len=64):")
print(f"  Original:  {original_time:.3f}s ({original_time/n_runs*1000:.1f} ms/run)")
print(f"  Quantized: {quantized_time:.3f}s ({quantized_time/n_runs*1000:.1f} ms/run)")


In [ ]:
#@title 🎧 Narration: Todo3 Post Quantization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_29_todo3_post_quantization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"  Speedup:   {original_time/quantized_time:.2f}x")

# YOUR CODE: Step 5 — Generate text from both models
prompt_ids = torch.tensor([encode("The ")], dtype=torch.long)

original_gen = model_cpu.generate(prompt_ids, max_new_tokens=150, temperature=0.8)
quantized_gen = model_q.generate(prompt_ids, max_new_tokens=150, temperature=0.8)

print(f"\n📝 Original model output:")
print(f"  {decode(original_gen[0].tolist())[:200]}")
print(f"\n📝 Quantized model output:")
print(f"  {decode(quantized_gen[0].tolist())[:200]}")

# YOUR CODE: Step 6 — Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Size comparison bar chart
???

# Speed comparison bar chart
???

plt.tight_layout()
plt.show()

---


In [ ]:
#@title 🎧 Narration: Stop And Think3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_30_stop_and_think3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### ✋ Stop and Think
Before looking at the solution:
1. Why does INT8 quantization give roughly 2-4x compression instead of exactly 4x (32/8)?
2. Dynamic quantization only quantizes the **weights**, not the activations. Why is this choice safer than quantizing both?
3. The GGUF Q4_K_M format achieves ~7x compression with minimal quality loss. What tricks does it use beyond simple uniform quantization?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Solution3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_31_solution3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
# Full quantization analysis

# Move to CPU for quantization
model_cpu = model.cpu()
model_cpu.eval()

# Step 1: Original model size
original_size_mb = sum(
    p.nelement() * p.element_size() for p in model_cpu.parameters()
) / 1e6
print(f"Original model size: {original_size_mb:.2f} MB")

# Step 2: Apply dynamic INT8 quantization
model_q = torch.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},  # Quantize all Linear layers
    dtype=torch.qint8,
)
print(f"✅ Dynamic INT8 quantization applied!")

# Step 3: Quantized model size
quantized_size_mb = sum(
    v.nelement() * v.element_size()
    for v in model_q.state_dict().values()
) / 1e6
print(f"Quantized model size: {quantized_size_mb:.2f} MB")
compression = original_size_mb / max(quantized_size_mb, 0.01)
print(f"Compression ratio: {compression:.2f}x")

# Step 4: Speed comparison
n_runs = 100
test_input = torch.randint(0, 256, (1, 64))

# Warm up
with torch.no_grad():
    for _ in range(5):
        _ = model_cpu(test_input)
        _ = model_q(test_input)

# Time original
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        _ = model_cpu(test_input)
original_time = time.time() - start

# Time quantized
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        _ = model_q(test_input)
quantized_time = time.time() - start

print(f"\nInference speed ({n_runs} forward passes, seq_len=64):")
print(f"  Original:  {original_time:.3f}s ({original_time/n_runs*1000:.1f} ms/run)")
print(f"  Quantized: {quantized_time:.3f}s ({quantized_time/n_runs*1000:.1f} ms/run)")
speedup = original_time / max(quantized_time, 0.001)
print(f"  Speedup:   {speedup:.2f}x")

# Step 5: Generate text from both
prompt_ids = torch.tensor([encode("The ")], dtype=torch.long)

original_gen = model_cpu.generate(prompt_ids.clone(), max_new_tokens=150, temperature=0.8)
quantized_gen = model_q.generate(prompt_ids.clone(), max_new_tokens=150, temperature=0.8)

print(f"\n📝 Original model output:")
print(f"  '{decode(original_gen[0].tolist())[:250]}'")
print(f"\n📝 Quantized model output:")
print(f"  '{decode(quantized_gen[0].tolist())[:250]}'")
print(f"\n(Both should produce similar quality — INT8 is nearly lossless)")

# Step 6: Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Size comparison
sizes = [original_size_mb, quantized_size_mb]
labels = ['FP32\n(Original)', 'INT8\n(Quantized)']
colors = ['#1565c0', '#4caf50']
bars = axes[0].bar(labels, sizes, color=colors, alpha=0.8, edgecolor='white', width=0.5)
axes[0].set_ylabel('Size (MB)', fontsize=12)
axes[0].set_title('Model Size', fontsize=14, fontweight='bold')
for bar, size in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{size:.1f} MB', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, max(sizes) * 1.3)

# Speed comparison
times = [original_time/n_runs*1000, quantized_time/n_runs*1000]
bars = axes[1].bar(labels, times, color=colors, alpha=0.8, edgecolor='white', width=0.5)
axes[1].set_ylabel('Time per Forward Pass (ms)', fontsize=12)
axes[1].set_title('Inference Speed', fontsize=14, fontweight='bold')
for bar, t in zip(bars, times):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{t:.1f} ms', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, max(times) * 1.3)

# Extrapolation: what Q4 would look like for real models
real_models = ['Qwen3.5\n9B', 'Qwen3.5\n35B', 'Qwen3.5\n122B', 'Qwen3.5\n397B']
fp16_sizes = [18, 70, 244, 794]
q4_sizes = [5, 20, 70, 230]

x = np.arange(len(real_models))
width = 0.35
axes[2].bar(x - width/2, fp16_sizes, width, color='#1565c0', alpha=0.8,
            label='FP16', edgecolor='white')
axes[2].bar(x + width/2, q4_sizes, width, color='#4caf50', alpha=0.8,
            label='Q4_K_M', edgecolor='white')
axes[2].set_ylabel('Size (GB)', fontsize=12)
axes[2].set_title('Real Qwen3.5: FP16 vs Q4', fontsize=14, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(real_models, fontsize=10)
axes[2].legend(fontsize=10)

# Annotate compression ratios
for i in range(len(real_models)):
    ratio = fp16_sizes[i] / q4_sizes[i]
    axes[2].annotate(f'{ratio:.1f}x',
                     xy=(x[i], max(fp16_sizes[i], q4_sizes[i])),
                     xytext=(0, 10), textcoords='offset points',
                     ha='center', fontsize=9, fontweight='bold', color='#e53935')

plt.tight_layout()
plt.show()

print(f"\n📊 Summary:")
print(f"  Our mini model: {original_size_mb:.1f} MB → {quantized_size_mb:.1f} MB ({compression:.1f}x compression)")
print(f"  Speed: {speedup:.1f}x faster with INT8")
print(f"\n  Real Qwen3.5-9B: ~18 GB (FP16) → ~5 GB (Q4_K_M) = 3.6x compression")
print(f"  That 5 GB file fits on any laptop with 8 GB RAM!")

---


In [ ]:
#@title 🎧 Transition: Part F Deployment Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_32_part_f_deployment_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part F: Local Deployment

### Running Qwen3.5 on Your Own Machine

The entire point of Qwen3.5's architecture is to make powerful AI run locally. Here is how it works in practice:

### Deployment Frameworks

| Framework | What It Is | Best For |
|-----------|-----------|----------|
| **Ollama** | One-command model serving | Easiest setup, great for beginners |
| **llama.cpp** | C/C++ inference engine | Maximum performance, custom builds |
| **LM Studio** | Desktop GUI application | Non-technical users, visual interface |

With Ollama, deploying Qwen3.5-9B is literally one command:
```bash
ollama run qwen3.5:9b
```

### The Hybrid Attention Advantage for Local Deployment

The 3:1 hybrid architecture provides a **critical advantage** for local deployment: reduced KV cache.

In a pure Transformer, every layer needs a KV cache that grows linearly with conversation length. For a 32-layer model processing 10,000 tokens at FP16:

$$\text{KV cache} = 2 \times 32 \times 10{,}000 \times 128 \times 2 = 163 \text{ MB per head}$$

With Qwen3.5's 3:1 pattern, only 25% of layers (the Gated Attention layers) need a KV cache. The Gated DeltaNet layers maintain a **fixed-size state** regardless of conversation length:

$$\text{DeltaNet state per layer} = d_k \times d_v \times 2 = 128 \times 128 \times 2 = 32 \text{ KB per head}$$

This means:
- **75% of layers**: Fixed memory (no growth with conversation length)
- **25% of layers**: Standard KV cache (grows, but only 1/4 of a pure Transformer)

The result: **4x less KV cache memory**, enabling longer conversations on limited hardware.


In [ ]:
#@title 🎧 Listen: Thinking Modes
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_33_thinking_modes.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Thinking Modes

Qwen3.5 supports three thinking modes:

| Mode | What Happens | Best For |
|------|-------------|----------|
| **Thinking** | Model reasons step-by-step before answering | Complex math, logic, analysis |
| **Non-thinking** | Model answers directly, no internal reasoning | Simple queries, speed-critical tasks |
| **Thinking budget** | Model gets N tokens for reasoning, then must answer | Balancing quality vs speed |

On a laptop, you might use:
- **Thinking mode** for a complex coding question (takes 10-30 seconds but gets it right)
- **Non-thinking mode** for a simple factual question (instant response)
- **Thinking budget of 500 tokens** for a moderate question (quick reasoning, then answer)

In [ ]:
#@title 🎧 What to Look For: Kv Cache Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_34_kv_cache_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# VISUALIZATION: KV Cache Savings from Hybrid Attention
# ============================================================

# Compare memory usage: pure Transformer vs 3:1 hybrid
seq_lengths = np.array([1000, 5000, 10000, 50000, 100000, 262144])

# Model parameters (Qwen3.5-9B-like)
n_layers = 36
n_heads = 28
d_head = 128
bytes_per_element = 2  # FP16

# Pure Transformer: all layers need KV cache
pure_kv_gb = []
for N in seq_lengths:
    # 2 (K+V) * n_layers * n_heads * N * d_head * bytes
    kv_bytes = 2 * n_layers * n_heads * N * d_head * bytes_per_element
    pure_kv_gb.append(kv_bytes / 1e9)

# Hybrid 3:1: only 25% of layers need KV cache, rest have fixed state
hybrid_kv_gb = []
n_attn_layers = n_layers // 4  # 9 layers with KV cache
n_delta_layers = n_layers - n_attn_layers  # 27 layers with fixed state

for N in seq_lengths:
    # KV cache for attention layers
    attn_kv_bytes = 2 * n_attn_layers * n_heads * N * d_head * bytes_per_element
    # Fixed state for DeltaNet layers (constant, does not grow with N)
    delta_state_bytes = n_delta_layers * n_heads * d_head * d_head * bytes_per_element
    hybrid_kv_gb.append((attn_kv_bytes + delta_state_bytes) / 1e9)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Absolute memory usage
axes[0].plot(seq_lengths / 1000, pure_kv_gb, 'o-', color='#e53935',
             linewidth=2.5, markersize=7, label='Pure Transformer (all softmax)')
axes[0].plot(seq_lengths / 1000, hybrid_kv_gb, 's-', color='#1565c0',
             linewidth=2.5, markersize=7, label='Qwen3.5 Hybrid (3:1 pattern)')
axes[0].fill_between(seq_lengths / 1000, hybrid_kv_gb, pure_kv_gb,
                     alpha=0.15, color='#4caf50')
axes[0].set_xlabel('Sequence Length (thousands of tokens)', fontsize=12)
axes[0].set_ylabel('KV Cache / State Memory (GB)', fontsize=12)
axes[0].set_title('Memory Savings from Hybrid Attention', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# Annotate the savings at the maximum
axes[0].annotate(
    f'{pure_kv_gb[-1] - hybrid_kv_gb[-1]:.1f} GB saved\n'
    f'({(1 - hybrid_kv_gb[-1]/pure_kv_gb[-1])*100:.0f}% reduction)',
    xy=(seq_lengths[-1]/1000, hybrid_kv_gb[-1]),
    xytext=(-80, 30), textcoords='offset points',
    fontsize=10, fontweight='bold', color='#4caf50',
    arrowprops=dict(arrowstyle='->', color='#4caf50'),
)

# Right: Memory breakdown at 262K tokens
labels = ['KV Cache\n(Attn layers)', 'Fixed State\n(DeltaNet layers)']
pure_vals = [pure_kv_gb[-1], 0]
hybrid_vals = [
    2 * n_attn_layers * n_heads * seq_lengths[-1] * d_head * bytes_per_element / 1e9,
    n_delta_layers * n_heads * d_head * d_head * bytes_per_element / 1e9,
]

x = np.arange(2)
width = 0.3
axes[1].bar(x - width/2, pure_vals, width, color='#e53935', alpha=0.8,
            label='Pure Transformer', edgecolor='white')
axes[1].bar(x + width/2, hybrid_vals, width, color='#1565c0', alpha=0.8,
            label='Qwen3.5 Hybrid', edgecolor='white')
axes[1].set_ylabel('Memory (GB)', fontsize=12)
axes[1].set_title(f'Memory Breakdown at {seq_lengths[-1]:,} tokens', fontsize=14,
                  fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, fontsize=11)
axes[1].legend(fontsize=10)

# Annotate
for i, (p, h) in enumerate(zip(pure_vals, hybrid_vals)):
    if p > 0:
        axes[1].text(x[i] - width/2, p + 0.2, f'{p:.1f} GB', ha='center', fontsize=9)
    if h > 0:
        axes[1].text(x[i] + width/2, h + 0.2, f'{h:.2f} GB', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nAt 262K tokens (a full book):")
print(f"  Pure Transformer KV cache: {pure_kv_gb[-1]:.1f} GB")
print(f"  Qwen3.5 hybrid memory:     {hybrid_kv_gb[-1]:.1f} GB")
print(f"  Savings:                    {pure_kv_gb[-1] - hybrid_kv_gb[-1]:.1f} GB ({(1 - hybrid_kv_gb[-1]/pure_kv_gb[-1])*100:.0f}%)")
print(f"\nThe DeltaNet state is tiny: {n_delta_layers * n_heads * d_head * d_head * bytes_per_element / 1e6:.1f} MB total")
print(f"  (constant regardless of sequence length!)")

---


In [ ]:
#@title 🎧 Transition: Part G Benchmarks Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_35_part_g_benchmarks_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part G: Benchmark Results — Qwen3.5 vs The World

The proof is in the benchmarks. Qwen3.5's architectural innovations translate to remarkable performance, especially considering the model sizes involved.

In [ ]:
#@title 🎧 What to Look For: Benchmark Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_36_benchmark_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# BENCHMARK RESULTS: Qwen3.5 vs Frontier Models
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Benchmark 1: GPQA Diamond (Graduate-level Science) ---
models_gpqa = ['GPT-OSS-120B', 'Claude-3.5\nSonnet', 'Llama-4\nMaverick', 'Qwen3.5-9B']
scores_gpqa = [71.5, 65.0, 69.8, 81.7]
colors_gpqa = ['#90a4ae', '#90a4ae', '#90a4ae', '#1565c0']

bars = axes[0].bar(models_gpqa, scores_gpqa, color=colors_gpqa, alpha=0.85,
                   edgecolor='white', width=0.6)
axes[0].set_ylabel('GPQA Diamond Score', fontsize=12)
axes[0].set_title('Graduate-Level Science\n(GPQA Diamond)', fontsize=13, fontweight='bold')
axes[0].set_ylim(55, 90)

for bar, score in zip(bars, scores_gpqa):
    color = '#1565c0' if score == max(scores_gpqa) else '#666'
    weight = 'bold' if score == max(scores_gpqa) else 'normal'
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{score}', ha='center', fontsize=12, fontweight=weight, color=color)

# Annotation
axes[0].annotate('9B model beats\n120B model!',
                 xy=(3, 81.7), xytext=(2.0, 86),
                 fontsize=9, fontweight='bold', color='#e53935',
                 arrowprops=dict(arrowstyle='->', color='#e53935'))

# --- Benchmark 2: BFCL-V4 (Tool Use) ---
models_bfcl = ['GPT-5 mini', 'Claude-3.5\nSonnet', 'Llama-4\nMaverick', 'Qwen3.5\n122B-A10B']
scores_bfcl = [55.5, 52.4, 61.2, 72.2]
colors_bfcl = ['#90a4ae', '#90a4ae', '#90a4ae', '#1565c0']

bars = axes[1].bar(models_bfcl, scores_bfcl, color=colors_bfcl, alpha=0.85,
                   edgecolor='white', width=0.6)
axes[1].set_ylabel('BFCL-V4 Score', fontsize=12)
axes[1].set_title('Tool Use Benchmark\n(BFCL-V4)', fontsize=13, fontweight='bold')
axes[1].set_ylim(40, 80)

for bar, score in zip(bars, scores_bfcl):
    color = '#1565c0' if score == max(scores_bfcl) else '#666'
    weight = 'bold' if score == max(scores_bfcl) else 'normal'
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{score}', ha='center', fontsize=12, fontweight=weight, color=color)

# --- Benchmark 3: Terminal-Bench 2.0 (Agentic Tasks) ---
models_tb = ['Qwen3\n(previous gen)', 'Claude-3.5\nSonnet', 'GPT-5 mini', 'Qwen3.5\n397B-A17B']
scores_tb = [22.5, 36.2, 38.1, 52.5]
colors_tb = ['#ffcc80', '#90a4ae', '#90a4ae', '#1565c0']

bars = axes[2].bar(models_tb, scores_tb, color=colors_tb, alpha=0.85,
                   edgecolor='white', width=0.6)
axes[2].set_ylabel('Terminal-Bench 2.0 Score', fontsize=12)
axes[2].set_title('Agentic Tasks\n(Terminal-Bench 2.0)', fontsize=13, fontweight='bold')
axes[2].set_ylim(0, 60)

for bar, score in zip(bars, scores_tb):
    color = '#1565c0' if score == max(scores_tb) else '#666'
    weight = 'bold' if score == max(scores_tb) else 'normal'
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{score}', ha='center', fontsize=12, fontweight=weight, color=color)

# Annotation for the improvement
axes[2].annotate('133% improvement\nvs previous gen',
                 xy=(0, 22.5), xytext=(0.8, 12),
                 fontsize=9, fontweight='bold', color='#e53935',
                 arrowprops=dict(arrowstyle='->', color='#e53935'))

plt.suptitle('Qwen3.5 Benchmark Performance', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("Key takeaways:")
print("  • Qwen3.5-9B (laptop-sized) beats GPT-OSS-120B on graduate-level science")
print("  • Qwen3.5-122B (MoE, activates only 10B) leads on tool use benchmarks")
print("  • Qwen3.5-397B shows 133% improvement over previous generation on agentic tasks")
print("  • All of this powered by the hybrid attention + MoE architecture we built!")

---


In [ ]:
#@title 🎧 Listen: Course Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_37_course_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 🎓 Course Reflection: What You Have Accomplished

Over five notebooks and one article, you built the complete Qwen3.5 architecture from first principles. Let us take a moment to appreciate the journey:

### Notebook 1: From Softmax to Linear Attention
You started with the $O(N^2)$ attention bottleneck and derived the **kernel trick** that makes attention linear. You saw how changing the order of matrix multiplication — the "right-to-left trick" — reduces the cost from $O(N^2 d)$ to $O(N d^2)$.

### Notebook 2: DeltaNet — Error-Correcting Memory
You discovered that naive linear attention is a "whiteboard with no eraser" and implemented the **delta rule** — a 1960s idea that turns a dumb accumulator into an error-correcting associative memory. You benchmarked it and saw orders-of-magnitude improvement.

### Notebook 3: Gated DeltaNet — The Full Layer
You added **exponential gating** to the delta rule, giving the model a "memory wipe button" for wholesale forgetting. You built the full Gated DeltaNet layer with Conv1D, SiLU, L2 normalization, and multi-head structure.

### Notebook 4: Hybrid Attention + MoE
You assembled the **3:1 hybrid pattern** — three cheap linear layers followed by one precise attention layer — and implemented **fine-grained Mixture-of-Experts** with 512 tiny experts and top-k routing.

### Notebook 5 (This One): Build & Run Mini-Qwen3.5
You put it all together into a **complete language model**, trained it from scratch, watched it learn to generate text, and quantized it to measure the real-world impact of compression.

### The Big Picture

What makes Qwen3.5 remarkable is not any single innovation — it is the **convergence** of ideas spanning decades:

| Decade | Idea | Role in Qwen3.5 |
|--------|------|-----------------|
| 1960s | Delta rule | Error-correcting memory in GatedDeltaNet |
| 1990s | Mixture-of-Experts | Parameter-efficient scaling (512 tiny experts) |
| 2017 | Self-attention | Gated Attention layers for global retrieval |
| 2020 | RoPE, RMSNorm | Stable training with long contexts |
| 2023 | State-space models | Inspiration for the recurrent GatedDeltaNet |
| 2024 | Gated DeltaNet | Core efficiency innovation (75% of layers) |
| 2026 | Qwen3.5 | Everything assembled into one architecture |

The result: a 9-billion parameter model that fits on a laptop and outperforms models 13 times its size. The age of AI being locked behind API calls and cloud servers is ending.

**You now understand exactly how it works — because you built it.**

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_38_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============================================================
# 🎉 COURSE COMPLETE!
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║   🎉 CONGRATULATIONS!                                           ║
║                                                                  ║
║   You have completed the entire                                  ║
║   "Build Qwen3.5 from Scratch" course!                          ║
║                                                                  ║
║   What you built:                                                ║
║   ├── Linear Attention (O(N) token mixing)                       ║
║   ├── Delta Rule (error-correcting memory)                       ║
║   ├── Gated DeltaNet (selective forgetting + correction)         ║
║   ├── 3:1 Hybrid Pattern (efficiency + precision)                ║
║   ├── Fine-grained MoE (512 experts, 10 active)                 ║
║   └── Complete Mini-Qwen3.5 (trained from scratch!)              ║
║                                                                  ║
║   Key numbers:                                                   ║
║   ├── 5M parameter mini model → trained on Shakespeare           ║
║   ├── 3:1 ratio → 75% linear + 25% full attention               ║
║   ├── INT8 quantization → ~2-4x size reduction                   ║
║   └── Real Qwen3.5-9B → 5 GB on any laptop                     ║
║                                                                  ║
║   The architecture that lets a 9B model outperform one          ║
║   13x its size — and you built every piece of it.               ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

# Final architecture summary
print("📋 Complete Architecture Cheat Sheet:")
print("""
┌─────────────────────────────────────────────────────────────┐
│                       QWEN3.5 ARCHITECTURE                  │
│                                                             │
│  Hybrid Attention (3:1 pattern):                            │
│    • GatedDeltaNet: S = α·S + β·k(v - S^T k)^T   [O(1)]  │
│    • GatedAttention: σ(g) ⊙ softmax(QK^T/√d)V     [O(N)]  │
│                                                             │
│  Feed-Forward (per-token):                                  │
│    • Dense: SwiGLU(x) = (SiLU(xW₁) ⊙ xW₃)W₂             │
│    • MoE: Top-10 of 512 experts + 1 shared expert           │
│                                                             │
│  Block structure:                                            │
│    output = RMSNorm(h + FFN(h))                             │
│    where h = RMSNorm(x + Attention(x))                      │
│                                                             │
│  Key efficiency gains:                                       │
│    • 75% of layers: O(1) per token (constant memory)        │
│    • 25% of layers: O(N) per token (KV cache, but sparse)   │
│    • MoE: 397B params, only 17B active per token            │
│    • Q4 quantization: 9B model → 5 GB on disk              │
└─────────────────────────────────────────────────────────────┘
""")

print("🚀 What to do next:")
print("  1. Try running Qwen3.5-9B locally with Ollama: ollama run qwen3.5:9b")
print("  2. Experiment with the mini model — try different hyperparameters!")
print("  3. Read the original Qwen3.5 paper for the full technical details")
print("  4. Check out the case study notebook for a real-world deployment scenario")
print()
print("Thank you for learning with Vizuara! 🙏")